# NeuSymNav M1–M5 RGB-D smoke — HM3D 00805

This notebook runs the pinned NeuSymNav M1–M5 path on fresh Habitat RGB-D frames. It does not use semantic ground truth or a synthetic map. The UI is disabled; review the exported M3/M5 map, hierarchy graph, run report, and M5 acceptance result below.

Select a Kaggle GPU accelerator and enable Internet for the first setup. The notebook installs CUDA-enabled PyTorch 2.7.1, uses CUDA for YOLOE inference and Triton navigation raycasting, and verifies the actual CUDA device before preflight. Habitat currently remains on the tested headless software-rendering path; Rebuild native M1–M5 mapping is CPU. The bounded smoke is 180 seconds; an M5 rejection or incomplete run remains rejected.

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import hashlib, json, os, shutil, subprocess, sys, tarfile, time, uuid
from IPython.display import Image, display


input_root = Path('/kaggle/input')
EXPECTED_SOURCE_TREE_SHA256 = '9eab6fd4bc96150edd5e2d0e4f4b32ecafd78b0b87715f590bba51b69f1110a7'
EXPECTED_DETECTOR_DIAGNOSTIC_SHA256 = '6ec70d2c2235dcd2169c574d24c1baa104c1aeb7582c5ad209649e03c2e9216a'
EXPECTED_NETWORK_PREFLIGHT_SHA256 = '75db083325f794450152c4a3f5558b6ec222e73a2ad4af93ec2ea6f72a84ce13'
EXPECTED_GPU_REQUIREMENTS_SHA256 = 'e70fbd2261057cd183908f2ce0c1997980342ab776ade8f792289cbfe3280eb6'
def sha256_path(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()
def matching_manifest_candidate(candidate):
    return (candidate.get('bundle') == 'neusymnav-rebuild-rgbd-m1-m5-v1'
            and candidate.get('map') == '00805'
            and candidate.get('kaggle_gpu_enabled') is True
            and candidate.get('kaggle_gpu_host_requirements_sha256') == EXPECTED_GPU_REQUIREMENTS_SHA256
            and candidate.get('detector_diagnostic_sha256') == EXPECTED_DETECTOR_DIAGNOSTIC_SHA256
            and candidate.get('network_preflight_sha256') == EXPECTED_NETWORK_PREFLIGHT_SHA256
            and candidate.get('neusymnav_source', {}).get('tree_sha256') == EXPECTED_SOURCE_TREE_SHA256)
def matching_manifests(root):
    found = []
    for path in root.glob('**/manifest.json'):
        try:
            candidate = json.loads(path.read_text())
        except (OSError, json.JSONDecodeError):
            continue
        if matching_manifest_candidate(candidate):
            found.append((path, candidate))
    return found

# Kaggle may expose an uploaded ZIP as an already-expanded dataset directory.
expanded = matching_manifests(input_root)
if len(expanded) == 1:
    manifest_path, manifest = expanded[0]
    bundle = manifest_path.parent
    bundle_source = 'expanded Kaggle dataset input'
elif expanded:
    raise RuntimeError(f'Expected one matching expanded bundle manifest; found {len(expanded)}')
else:
    archive_matches = []
    for candidate_zip in input_root.glob('**/*.zip'):
        try:
            with ZipFile(candidate_zip) as candidate_archive:
                manifest_names = [name for name in candidate_archive.namelist() if name.endswith('/manifest.json') or name == 'manifest.json']
                for manifest_name in manifest_names:
                    candidate_manifest = json.loads(candidate_archive.read(manifest_name))
                    if matching_manifest_candidate(candidate_manifest):
                        archive_matches.append(candidate_zip)
                        break
        except (OSError, ValueError, json.JSONDecodeError, KeyError):
            continue
    if len(archive_matches) != 1:
        raise RuntimeError(f'Expected one archived GPU bundle pinned to the detector diagnostic helper; found {len(archive_matches)}')
    input_zip = archive_matches[0]
    extracted = Path('/kaggle/working/rebuild-rgbd-00805-full-depthfix-input')
    extracted.mkdir(parents=True, exist_ok=True)
    with ZipFile(input_zip) as zf:
        for name in zf.namelist():
            member = Path(name)
            if member.is_absolute() or '..' in member.parts:
                raise ValueError(f'Unsafe path in input archive: {name}')
        zf.extractall(extracted)
    extracted_manifests = matching_manifests(extracted)
    if len(extracted_manifests) != 1:
        raise RuntimeError(f'Expected one matching bundle manifest in ZIP; found {len(extracted_manifests)}')
    manifest_path, manifest = extracted_manifests[0]
    bundle = manifest_path.parent
    bundle_source = 'uploaded ZIP extracted in working storage'

for relative_path, expected_hash in (('probe/detector_diagnostic.py', EXPECTED_DETECTOR_DIAGNOSTIC_SHA256),
                            ('probe/network_preflight.py', EXPECTED_NETWORK_PREFLIGHT_SHA256),
                            ('requirements-host-kaggle-gpu.txt', EXPECTED_GPU_REQUIREMENTS_SHA256)):
    actual_hash = sha256_path(bundle/relative_path)
    if actual_hash != expected_hash:
        raise RuntimeError(f'Bundle file pin mismatch for {relative_path}: {actual_hash}')

print('bundle:', bundle)
print('bundle source:', bundle_source)
print('manifest:', json.dumps(manifest, indent=2))
print('kernel:', sys.version.split()[0], sys.executable)
print('working free GiB:', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))
print('GPU probe:', subprocess.run(['bash','-lc','nvidia-smi'],capture_output=True,text=True).stdout[:1200] or 'No NVIDIA GPU visible')

In [ ]:
# Create the exact split environments: NeuSymNav/detector on Python 3.12, Habitat on 3.9.
network_preflight = subprocess.run([str(sys.executable),str(bundle/'probe/network_preflight.py'),
    '--timeout-s','2'],capture_output=True,text=True)
print('Kaggle bootstrap network preflight:')
print(network_preflight.stdout or network_preflight.stderr)
if network_preflight.returncode:
    raise RuntimeError('Kaggle DNS/TCP preflight failed; restore kernel Internet connectivity before environment setup.')
manager = shutil.which('conda') or shutil.which('micromamba')
if manager is None:
    import io, urllib.request
    archive = urllib.request.urlopen('https://micro.mamba.pm/api/micromamba/linux-64/latest', timeout=90).read()
    with tarfile.open(fileobj=io.BytesIO(archive), mode='r:bz2') as tar:
        target = Path('/kaggle/working/bin/micromamba')
        target.parent.mkdir(parents=True, exist_ok=True)
        member = tar.getmember('bin/micromamba')
        target.write_bytes(tar.extractfile(member).read())
        target.chmod(0o755)
    manager = str(target)
mamba_env = dict(os.environ, MAMBA_ROOT_PREFIX='/kaggle/working/mamba-root')
host_env = Path('/kaggle/working/envs/neusymnav-host-312')
habitat_env = Path('/kaggle/working/envs/habitat-039')
host_python = host_env / 'bin/python'
habitat_python = habitat_env / 'bin/python'
if not host_python.exists():
    subprocess.run([manager,'create','-y','-p',str(host_env),'python=3.12','pip','-c','conda-forge'],check=True,env=mamba_env)
if not habitat_python.exists():
    subprocess.run([manager,'create','-y','-p',str(habitat_env),'python=3.9','habitat-sim=0.3.3','headless','-c','conda-forge','-c','aihabitat'],check=True,env=mamba_env)
# Install the bundle's CUDA-enabled, pinned host environment (including Triton).
subprocess.run([str(host_python),'-m','pip','install','--disable-pip-version-check','--no-cache-dir',
    '--extra-index-url','https://download.pytorch.org/whl/cu126',
    '-r',str(bundle/'requirements-host-kaggle-gpu.txt')],check=True)
# Import the bundled source directly; avoid an isolated wheel build for this source-only package.
# Kaggle expands this bundled .txt.gz asset to a plain .txt input file.
# Build the import tree in writable storage and restore the filename/encoding expected by CLIP.
import gzip
clip_import_root = Path('/kaggle/working/rebuild-rgbd-00805-full/clip-import')
if not clip_import_root.exists():
    shutil.copytree(bundle/'project/third_party/clip', clip_import_root)
clip_package = clip_import_root/'clip'
clip_vocab_gz = clip_package/'bpe_simple_vocab_16e6.txt.gz'
clip_vocab_txt = clip_package/'bpe_simple_vocab_16e6.txt'
if not clip_vocab_gz.exists() and clip_vocab_txt.is_file():
    raw_vocab = clip_vocab_txt.read_bytes()
    try:
        gzip.decompress(raw_vocab)
        gzip_payload = raw_vocab
    except OSError:
        gzip_payload = gzip.compress(raw_vocab, mtime=0)
    clip_vocab_gz.write_bytes(gzip_payload)
if not clip_vocab_gz.is_file():
    raise FileNotFoundError(f'CLIP tokenizer vocabulary missing: {clip_vocab_gz}')
subprocess.run([str(host_python),'-m','pip','install','--disable-pip-version-check','--no-build-isolation','--no-deps',str(clip_import_root)],check=True)
clip_probe_env = dict(os.environ, PYTHONPATH=str(clip_import_root))
versions = subprocess.run([str(host_python),'-c',"import sys,torch,numpy,scipy,ultralytics,clip; import habitat_sim; print(sys.version.split()[0], torch.__version__, numpy.__version__, scipy.__version__, ultralytics.__version__, clip.__file__)"],capture_output=True,text=True,env=clip_probe_env)
if versions.returncode:
    print('host import probe stderr:', versions.stderr)
    # Habitat is intentionally isolated in the Python 3.9 process.
    versions = subprocess.run([str(host_python),'-c',"import sys,torch,numpy,scipy,ultralytics,clip; print(sys.version.split()[0], torch.__version__, numpy.__version__, scipy.__version__, ultralytics.__version__, clip.__file__)"],check=True,capture_output=True,text=True,env=clip_probe_env)
habitat_probe = subprocess.run([str(habitat_python),'-c',"import habitat_sim; print(habitat_sim.__version__, habitat_sim.cuda_enabled)"],check=True,capture_output=True,text=True)
print('host:', versions.stdout.strip())
print('Habitat:', habitat_probe.stdout.strip())
gpu_probe = subprocess.run([str(host_python),'-c',
    "import json,torch,triton; assert torch.cuda.is_available() and torch.version.cuda; \
print(json.dumps({'torch':torch.__version__,'cuda_runtime':torch.version.cuda,\
'gpu':torch.cuda.get_device_name(0),'triton':triton.__version__}))"],
    check=True,capture_output=True,text=True)
print('GPU execution environment:', gpu_probe.stdout.strip())

In [ ]:
project = bundle / 'project'
config = Path('/kaggle/working/rebuild-rgbd-00805-full/runtime.json')
configure = subprocess.run([str(host_python),str(bundle/'probe/configure_runtime.py'),
    '--bundle',str(bundle),'--host-python',str(host_python),'--habitat-python',str(habitat_python),
    '--detector-device','cuda:0','--raycast-backend','triton_cuda',
    '--output',str(config)],capture_output=True,text=True)
print(configure.stdout)
if configure.returncode:
    raise RuntimeError(configure.stderr)
configured = json.loads(config.read_text())
if configured['detector']['device'] != 'cuda:0' or configured['navigation']['mapper']['raycast_backend'] != 'triton_cuda':
    raise RuntimeError('GPU stages were not selected explicitly in the runtime profile')
native_bin_dir = Path('/kaggle/working/rebuild-rgbd-00805-full/native-bin')
native_bin_dir.mkdir(parents=True, exist_ok=True)
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()
def make_executable(spec, name):
    source = Path(spec['path'])
    target = native_bin_dir/name
    shutil.copy2(source, target)
    target.chmod(target.stat().st_mode | 0o111)
    spec['path'] = str(target)
    spec['sha256'] = sha256_file(target)
# Kaggle working storage is smaller than the source profile's 32 GiB reservation.
# Keep this 180 s diagnostic bounded to explicit recording caps.
configured.setdefault('budgets',{}).update({'raw_cap':2*1024**3,'total_cap':3*1024**3,'archive_cap':256*1024**2})
config.write_text(json.dumps(configured,indent=2,sort_keys=True)+'\n')
room = configured['workers']['room_pipeline']
make_executable(configured['workers']['region']['executable'], 'occusg_dude_region_worker')
make_executable(room['m23_binary'], 'standalone_sg_octomap_fixture_runner')
make_executable(room['m4_binary'], 'occusg_dude_m4_worker')
config.write_text(json.dumps(configured,indent=2,sort_keys=True)+'\n')
env = dict(os.environ)
env.update({
    'PYTHONPATH':str(project/'src')+os.pathsep+str(clip_import_root),
    'LD_LIBRARY_PATH':str(bundle/'native/lib')+os.pathsep+str(project/'local-assets/native/lib')+os.pathsep+env.get('LD_LIBRARY_PATH',''),
    'EGL_PLATFORM':'surfaceless','LIBGL_ALWAYS_SOFTWARE':'1','PYTHONDONTWRITEBYTECODE':'1',
    'OPENBLAS_NUM_THREADS':'1','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1','NUMBA_NUM_THREADS':'1',
    'YOLO_AUTOINSTALL':'false','XDG_CACHE_HOME':'/kaggle/working/cache',
    'TORCH_HOME':'/kaggle/working/cache/torch','YOLO_CONFIG_DIR':'/kaggle/working/cache/ultralytics',
})
ldd = subprocess.run(['ldd',configured['workers']['room_pipeline']['m4_binary']['path']],env=env,capture_output=True,text=True,check=True)
print('M4 dynamic libraries:')
print(ldd.stdout)
if 'not found' in ldd.stdout:
    raise RuntimeError('Pinned M4 worker has unresolved runtime libraries')
preflight = subprocess.run([str(host_python),'-m','neusymnav.cli','preflight','--config',str(config)],env=env,capture_output=True,text=True)
Path('/kaggle/working/rebuild-rgbd-00805-full').mkdir(parents=True,exist_ok=True)
Path('/kaggle/working/rebuild-rgbd-00805-full/preflight.json').write_text(preflight.stdout or preflight.stderr)
print(preflight.stdout)
if preflight.returncode:
    raise RuntimeError(f'NeuSymNav preflight failed (exit {preflight.returncode})')
print('preflight: ready')

In [ ]:
# One bounded live run; retain all admission certificates and resource diagnostics.
run_seconds = int(os.environ.get('NEUSYMNAV_RUN_SECONDS','180'))
maximum_run_seconds = int(configured.get('recording_policy',{}).get('max_duration_s',0))
if not 1 <= run_seconds <= maximum_run_seconds:
    raise ValueError(f'run duration must be 1..{maximum_run_seconds} seconds for this pinned profile')
output = Path('/kaggle/working/rebuild-rgbd-00805-full')
run_id = time.strftime('%Y%m%dT%H%M%SZ',time.gmtime())+'-'+uuid.uuid4().hex[:8]
run_root = output/'runs'/f'{run_id}-{run_seconds}s'
run_root.mkdir(parents=True,exist_ok=False)
run_dir = run_root/'run'
started = time.monotonic()
run = subprocess.run([str(host_python),'-m','neusymnav.cli','run','--config',str(config),
    '--duration-s',str(run_seconds),'--out',str(run_dir)],env=env,capture_output=True,text=True,timeout=run_seconds+240)
elapsed = time.monotonic() - started
(run_root/'run.stdout.log').write_text(run.stdout)
(run_root/'run.stderr.log').write_text(run.stderr)
summary_path = run_dir / 'summary.json'
if not summary_path.is_file():
    print(run.stderr[-6000:])
    raise RuntimeError(f'Run produced no summary.json (exit {run.returncode})')
summary = json.loads(summary_path.read_text())
detector_diagnostic_path = run_root / 'detector-diagnostic.json'
try:
    detector_diagnostic = subprocess.run([str(host_python),str(bundle/'probe/detector_diagnostic.py'),
        '--run-dir',str(run_dir),'--worker-script',str(project/'src/neusymnav/perception/detector_worker.py'),
        '--output',str(detector_diagnostic_path),'--timeout-s','120'],env=env,capture_output=True,text=True,timeout=420)
except subprocess.TimeoutExpired as exc:
    detector_diagnostic = subprocess.CompletedProcess([],124,exc.stdout or '',exc.stderr or 'diagnostic outer deadline exceeded')
    detector_diagnostic_path.write_text(json.dumps({'status':'timed_out','error':'diagnostic outer deadline exceeded',
        'diagnostic_only':True,'production_deadline_changed':False},indent=2)+'\n')
(run_root/'detector-diagnostic.stdout.log').write_text(str(detector_diagnostic.stdout or ''))
(run_root/'detector-diagnostic.stderr.log').write_text(str(detector_diagnostic.stderr or ''))
detector_probe = json.loads(detector_diagnostic_path.read_text()) if detector_diagnostic_path.is_file() else {
    'status':'unavailable','error':f'diagnostic process exited {detector_diagnostic.returncode} without report'}
print('isolated detector diagnostic exit:',detector_diagnostic.returncode)
print('isolated detector diagnostic:',json.dumps(detector_probe,indent=2))
runtime = summary.get('runtime',{})
metrics = runtime.get('metrics',{})
room_metrics = metrics.get('room_pipeline',{})
room_counts = room_metrics.get('counts',{})
room_wire_path = run_dir / 'room-pipeline.json'
room_wire = json.loads(room_wire_path.read_text()) if room_wire_path.is_file() else {}
room_timings = room_wire.get('payload',{}).get('fields',{}).get('timings_ms',{})
resource_cleanup = summary.get('cleanup',{}).get('resources',{})
print(json.dumps({
    'run_exit_code':run.returncode,'elapsed_wall_s':round(elapsed,2),
    'outcome':summary.get('outcome'),'sensor_frames':summary.get('frames'),
    'raw_recording_certificate':summary.get('recording_certificate'),
    'runtime_clean_shutdown':runtime.get('clean_shutdown'),
    'graph_recorder_certificate':runtime.get('recorder_certificate'),
    'graph_scheduler':metrics.get('scheduler'),
    'semantic_sampling':runtime.get('semantic_sampling'),
    'room_queue':{'counts':room_counts,'queue_byte_cap':room_metrics.get('queue_byte_cap'),
                  'timings_ms_latest_update':room_timings},
    'room_pipeline_revisions':summary.get('room_pipeline_revisions'),
    'worker_runtime':summary.get('worker_runtime'),
    'resource_cleanup':{'complete':resource_cleanup.get('complete'),
        'rss':resource_cleanup.get('rss'),'gpu':resource_cleanup.get('gpu')},
    'resource_provider_diagnostics':summary.get('resource_provider_diagnostics'),
    'error':summary.get('error'),'admission_stop':summary.get('admission_stop')
},indent=2))

In [ ]:
# Export actual saved M3/M5 and hierarchy snapshots and evaluate the M5 acceptance contract.
review_dir = run_root / 'review'
export = subprocess.run([str(host_python),str(bundle/'probe/export_run.py'),str(run_dir),
    '--project-root',str(project),'--output',str(review_dir)],env=env,capture_output=True,text=True)
(run_root/'export.stdout.log').write_text(export.stdout)
(run_root/'export.stderr.log').write_text(export.stderr)
review_path = review_dir / 'review.json'
if review_path.is_file():
    review = json.loads(review_path.read_text())
    print(json.dumps(review,indent=2))
    if (review_dir/'rebuild_m3_m5.png').is_file(): display(Image(filename=str(review_dir/'rebuild_m3_m5.png')))
    if (review_dir/'hierarchy_graph.png').is_file(): display(Image(filename=str(review_dir/'hierarchy_graph.png')))
    print('M5 accepted:', review.get('acceptance',{}).get('accepted'))
else:
    print(export.stderr[-6000:])
    raise RuntimeError(f'Exporter produced no review.json (exit {export.returncode})')

In [ ]:
# Persist the diagnostics and acceptance result, then archive all run evidence.
output = Path('/kaggle/working/rebuild-rgbd-00805-full')
run_summary = json.loads((run_dir/'summary.json').read_text())
review = json.loads((review_dir/'review.json').read_text())
measurement_path = run_dir/'measurements.json'
measurement = json.loads(measurement_path.read_text()) if measurement_path.is_file() else {}
room_wire_path = run_dir / 'room-pipeline.json'
room_timing = {}
if room_wire_path.is_file():
    room_timing = json.loads(room_wire_path.read_text()).get('payload',{}).get('fields',{}).get('timings_ms',{})
runtime = run_summary.get('runtime',{})
metrics = runtime.get('metrics',{})
worker_runtime = run_summary.get('worker_runtime',{})
detector_runtime = worker_runtime.get('detector',{})
raycast_runtime = worker_runtime.get('raycast',{})
predict_events = [event for event in detector_probe.get('worker_events',[]) if event.get('stage') == 'model_predict']
stage_devices = {
    'detector':{'configured_device':detector_runtime.get('actual_recipe',{}).get('device'),
                'probe_status':detector_probe.get('status'),
                'observed_device':predict_events[-1].get('observed_device') if detector_probe.get('status') == 'completed' and predict_events else None},
    'raycast':{'backend':raycast_runtime.get('backend'),'status':raycast_runtime.get('status'),
               'observed_device':raycast_runtime.get('execution',{}).get('device_name'),
               'requests_completed':raycast_runtime.get('requests_completed')},
    'habitat_renderer':'Mesa llvmpipe (CPU)','rebuild_m1_m5':'native CPU'}
result = {
    'bundle':manifest['bundle'],'map':manifest['map'],'preflight_exit_code':preflight.returncode,
    'run_id':run_id,'run_seconds_requested':run_seconds,'run_dir':str(run_dir),
    'run_exit_code':run.returncode,'run_outcome':run_summary.get('outcome'),
    'acceptance':review.get('acceptance'),'m3_m5':review.get('rebuild_map'),
    'room_session':review.get('room_session'),'hierarchy':review.get('hierarchy_image'),
    'elapsed_wall_s':round(elapsed,2),'sensor_frames':run_summary.get('frames'),
    'raw_recording_complete':run_summary.get('recording_certificate',{}).get('complete'),
    'runtime_clean_shutdown':runtime.get('clean_shutdown'),
    'graph_recorder_certificate':runtime.get('recorder_certificate'),
    'graph_scheduler':metrics.get('scheduler'),
    'room_queue_counts':metrics.get('room_pipeline',{}).get('counts'),
    'room_stage_timings_ms_latest':room_timing,
    'measurement_completion':measurement.get('completion'),
    'resource_cleanup_complete':run_summary.get('cleanup',{}).get('resources',{}).get('complete'),
    'resource_provider_diagnostics':run_summary.get('resource_provider_diagnostics'),
    'stage_devices':stage_devices,'detector_diagnostic':detector_probe,
}
result_path = run_root/'kaggle_result.json'
result_path.write_text(json.dumps(result,indent=2,sort_keys=True)+'\n')
archive = run_root/f'rebuild-rgbd-00805-m1-m5-results-{run_seconds}s.zip'
with ZipFile(archive,'w',ZIP_DEFLATED,compresslevel=1) as zf:
    for path in sorted(run_root.rglob('*')):
        if path.is_file() and path != archive:
            zf.write(path,path.relative_to(run_root).as_posix())
print(json.dumps(result,indent=2))
print('result archive:',archive)
from IPython.display import FileLink
display(FileLink(str(archive.relative_to(Path('/kaggle/working')))))